# 01 - Data collection

Source inventory and provenance only. Reusable logic lives in `src/source_registry.py` and `src/collection_validation.py`; this notebook reports registered immutable sources and does not perform new collection, preparation, repair, or analysis.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.collection_validation import validate_icnf_archive, validate_zip_archive
from src.config import PILOT_2023_TO_2024
from src.source_registry import CAOP_2025, ICNF_2024, PILOT_ICNF_ARCHIVES

print(f'Project root: {PROJECT_ROOT}')
print('No automated downloads are executed.')

## Registered immutable archives

ZIP checks occur in place. ICNF Shapefile components are extracted only to a system temporary directory for read-only inspection. Invalid geometries are reported as source facts; they are not repaired, dropped, or interpreted as no-fire observations.

In [ ]:
caop_2025_validation = validate_zip_archive(CAOP_2025, PROJECT_ROOT)
print('CAOP 2025 ZIP:', caop_2025_validation)

pilot_icnf_validation = {}
for year, record in PILOT_ICNF_ARCHIVES.items():
    pilot_icnf_validation[year] = validate_icnf_archive(record, PROJECT_ROOT, expected_year=year)

for year, result in pilot_icnf_validation.items():
    print(
        f'{year}: {result["feature_count"]} features; '
        f'{result["non_empty_geometry_count"]} non-empty; '
        f'{result["invalid_geometry_count"]} invalid geometries'
    )

## 2023 to 2024 pilot archive roles

The ICNF history is 2013 through 2022. The 2024 archive is the observed outcome. ICNF 2023 is deliberately not a pilot predictor archive; same-year burned area is never a predictor.

In [ ]:
history_years = tuple(year for year in PILOT_ICNF_ARCHIVES if year != PILOT_2023_TO_2024.outcome_year)
assert history_years == PILOT_2023_TO_2024.historical_fire_years
assert PILOT_2023_TO_2024.predictor_year not in PILOT_ICNF_ARCHIVES

print(f'Predictor year: {PILOT_2023_TO_2024.predictor_year}')
print(f'Historical ICNF years: {history_years[0]}-{history_years[-1]}')
print(f'Outcome ICNF year: {PILOT_2023_TO_2024.outcome_year}')
print('No same-year ICNF predictor archive is registered for the pilot.')